# 04 — Historical profiles

AIDP operational pipeline notebook. It uses Volumes and governed tables only; the Git checkout is code only.

In [ ]:
from pathlib import Path
import os
import sys
from pyspark.sql import SparkSession, functions as F, Window
spark = SparkSession.builder.getOrCreate()
root = Path.cwd().resolve()
for candidate in (Path(os.environ['AIDP_PROJECT_ROOT']).resolve(), root, *root.parents) if os.getenv('AIDP_PROJECT_ROOT') else (root, *root.parents):
    if (candidate / 'src').is_dir() and (candidate / 'config/settings.yaml').is_file():
        root = candidate
        break
else:
    raise RuntimeError('Project code root not found. Clone the repository; do not use it as a data store.')
sys.path.insert(0, str(root / 'src'))
from aidp_pipeline import LANDING_COUNTS, RAW_SESSIONS, STRUCTURAL_KEYS, landing_paths, load_aidp_config, missing_landing_paths, table_name
config = load_aidp_config(root / 'config/settings.yaml')


In [ ]:
race = spark.table(table_name(config, 'silver_schema', 'race_state'))
valid = race.filter((F.col('lap_time_seconds') > 0) & (~F.coalesce(F.col('is_pit_lap'), F.lit(False))))
# Validated local method: retain positive non-pit laps within 4 × session MAD.
medians = valid.groupBy('session').agg(F.expr('percentile_approx(lap_time_seconds, 0.5, 10000)').alias('_median'))
with_median = valid.join(medians, 'session').withColumn('_abs_dev', F.abs(F.col('lap_time_seconds') - F.col('_median')))
mads = with_median.groupBy('session').agg(F.expr('percentile_approx(_abs_dev, 0.5, 10000)').alias('_mad'))
robust = with_median.join(mads, 'session').filter(F.col('_abs_dev') <= 4 * F.col('_mad'))
profiles = robust.filter(F.col('session').isin('R1','R2')).groupBy('driver_id').agg(
    F.count('*').alias('pooled_valid_laps'), F.expr('percentile_approx(lap_time_seconds, 0.5, 10000)').alias('pooled_median_pace'),
    F.expr('percentile_approx(_abs_dev, 0.5, 10000)').alias('pooled_MAD'))
profiles = profiles.withColumn('support_status', F.when(F.col('pooled_valid_laps') >= 5, F.lit('sufficient')).otherwise(F.lit('limited')))
historical_summary = robust.groupBy('session','driver_id').agg(F.count('*').alias('valid_lap_count'), F.expr('percentile_approx(lap_time_seconds, 0.5, 10000)').alias('median_lap_time'))
variability = robust.filter(F.col('session').isin('R1','R2')).join(historical_summary.select('session','driver_id',F.col('median_lap_time').alias('_driver_pace')), ['session','driver_id']).withColumn('lap_deviation', F.col('lap_time_seconds') - F.col('_driver_pace')).select('session','driver_id','lap_number','lap_deviation')


In [ ]:
catalog = config["catalog"]
silver = config["silver_schema"]
gold_hist = config["gold_historical_schema"]
gold_scenario = config["gold_scenario_schema"]

print("Configured schemas:")
print("Silver:", f"{catalog}.{silver}")
print("Gold Historical:", f"{catalog}.{gold_hist}")
print("Gold Scenario:", f"{catalog}.{gold_scenario}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_hist}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_scenario}")

print("SCHEMAS READY")

In [ ]:
for name, frame in [
    ("observed_driver_profiles", profiles),
    ("historical_pace_summary", historical_summary),
    ("historical_variability", variability)
]:
    frame.write.mode("overwrite").saveAsTable(
        table_name(config, "gold_historical_schema", name)
    )

In [ ]:
for name in [
    "observed_driver_profiles",
    "historical_pace_summary",
    "historical_variability",
]:
    target = table_name(config, "gold_historical_schema", name)
    print("\n", target)
    spark.table(target).show(5, truncate=False)
    print("rows:", spark.table(target).count())

In [ ]:
ranked = (
    spark.table(
        table_name(
            config,
            "gold_historical_schema",
            "observed_driver_profiles"
        )
    )
    .filter("support_status = 'sufficient'")
    .orderBy("pooled_median_pace")
    .limit(2)
    .collect()
)

gap = float(ranked[1].pooled_median_pace - ranked[0].pooled_median_pace)

print("P1/P2 robust pace gap:", gap)